# EEG Mental Workload Classification — STEW Dataset
### INEL 4998 | UPRM AIIG Lab | Favian O. Díaz
**Ejecuta cada celda en orden, de arriba a abajo (Shift+Enter o el botón ▶️)**

## Celda 1 — Conectar Google Drive
Ejecuta esto y acepta el permiso cuando aparezca el popup. Puedes usar **cualquier cuenta de Google**, no tiene que ser la misma de Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive conectado correctamente")

## Celda 2 — Verificar que encontramos el dataset
Esto busca la carpeta 'STEW Dataset' en tu Drive y muestra los primeros archivos.

In [ ]:
import os, glob

# Ajusta este path si tu carpeta tiene un nombre diferente
# Ejemplos comunes:
#   "/content/drive/MyDrive/STEW Dataset"
#   "/content/drive/MyDrive/STEW_data"
DATA_DIR = "/content/drive/MyDrive/STEW Dataset"

if os.path.exists(DATA_DIR):
    files = sorted(os.listdir(DATA_DIR))
    print(f"✓ Carpeta encontrada: {DATA_DIR}")
    print(f"  Total de archivos: {len(files)}")
    print(f"  Primeros 6: {files[:6]}")
else:
    # Buscar automaticamente en todo el Drive
    print("Carpeta no encontrada en el path por defecto. Buscando...")
    results = glob.glob('/content/drive/**/*sub01*', recursive=True)
    if results:
        DATA_DIR = os.path.dirname(results[0])
        print(f"✓ Encontrado automaticamente en: {DATA_DIR}")
    else:
        print("✗ No se encontro la carpeta.")
        print("  Asegurate de que el Drive este montado y la carpeta este ahi.")
        print("  Edita DATA_DIR arriba con el path correcto.")
        
print(f"\nDATA_DIR = '{DATA_DIR}'"  )

## Celda 3 — Instalar dependencias
Solo tarda unos segundos, Colab ya tiene la mayoría.

In [ ]:
!pip install scipy scikit-learn matplotlib seaborn -q
print('✓ Dependencias listas')

## Celda 4 — Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal as sp_signal
from scipy.stats import entropy, skew
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Configuración ──────────────────────────────────────────────────────────
FS             = 128
N_CHANNELS     = 14
CHANNEL_NAMES  = ['AF3','F7','F3','FC5','T7','P7','O1',
                  'O2','P8','T8','FC6','F4','F8','AF4']
WINDOW_SEC     = 2
WINDOW_SAMPLES = WINDOW_SEC * FS   # 256 samples
OVERLAP        = 0.5
BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 45)
}
SEED = 42
np.random.seed(SEED)
print("✓ Imports y configuracion listos")

## Celda 5 — Cargar el dataset STEW
Lee todos los sujetos de la carpeta. Con 48 sujetos puede tardar ~30 segundos.

In [ ]:
def load_stew_dataset(data_dir):
    subjects = []
    files = os.listdir(data_dir)
    subject_ids = sorted(set(f.split('_')[0] for f in files
                             if f.endswith('.txt') and f != 'ratings.txt'))
    print(f"Sujetos encontrados: {len(subject_ids)}")
    for sub_id in subject_ids:
        lo_file = os.path.join(data_dir, f"{sub_id}_lo.txt")
        hi_file = os.path.join(data_dir, f"{sub_id}_hi.txt")
        if not (os.path.exists(lo_file) and os.path.exists(hi_file)):
            continue
        try:
            lo = np.loadtxt(lo_file)[:19200, :N_CHANNELS]
            hi = np.loadtxt(hi_file)[:19200, :N_CHANNELS]
            subjects.append({'id': sub_id, 'rest': lo, 'task': hi})
            print(f"  ✓ {sub_id} cargado  ({lo.shape[0]} samples x {lo.shape[1]} canales)")
        except Exception as e:
            print(f"  ✗ Error en {sub_id}: {e}")
    return subjects

subjects = load_stew_dataset(DATA_DIR)
print(f"\n✓ Total: {len(subjects)} sujetos cargados")

## Celda 6 — Preprocesamiento
Filtrado, remoción de DC y rechazo de artefactos.

In [ ]:
def bandpass_filter(data, low, high, fs=FS, order=4):
    nyq = fs / 2.0
    b, a = sp_signal.butter(order, [low/nyq, high/nyq], btype='band')
    return sp_signal.filtfilt(b, a, data, axis=0)

def preprocess_eeg(eeg):
    eeg = eeg - eeg.mean(axis=0)          # DC removal
    eeg = bandpass_filter(eeg, 1.0, 45.0) # Bandpass 1-45 Hz
    # Artifact rejection: reemplaza muestras > 100 uV con interpolacion
    mask = np.any(np.abs(eeg) > 100.0, axis=1)
    if mask.sum() > 0:
        clean = np.where(~mask)[0]
        for ch in range(eeg.shape[1]):
            eeg[mask, ch] = np.interp(np.where(mask)[0], clean, eeg[clean, ch])
    return eeg

print("✓ Funciones de preprocesamiento definidas")

## Celda 7 — Extracción de features
Implementa exactamente el pipeline de la Sección 3.5 del proposal:
- Potencia espectral en theta, alpha, beta por canal
- **Ratio frontal θ / parietal α** (proxy del PCLI)
- Entropía espectral
- Estadísticas temporales

In [ ]:
def compute_band_power(epoch, fs=FS):
    nperseg = min(epoch.shape[0], 128)
    freqs, psd = sp_signal.welch(epoch, fs=fs, nperseg=nperseg, axis=0)
    band_powers = {}
    for name, (lo, hi) in BANDS.items():
        idx = np.where((freqs >= lo) & (freqs < hi))[0]
        if len(idx) == 0:
            band_powers[name] = np.zeros(epoch.shape[1])
        else:
            band_powers[name] = np.trapz(psd[idx,:], freqs[idx], axis=0) if hasattr(np, 'trapz')                                  else np.trapezoid(psd[idx,:], freqs[idx], axis=0)
    return band_powers, freqs, psd

def extract_features(epoch):
    features, names = [], []
    bp, freqs, psd = compute_band_power(epoch)
    total = sum(bp[b] for b in BANDS)
    total = np.where(total == 0, 1e-10, total)

    for band in ['theta','alpha','beta']:
        for ch, ch_name in enumerate(CHANNEL_NAMES):
            features.append(np.log1p(bp[band][ch]))
            names.append(f'abs_{band}_{ch_name}')

    for band in ['theta','alpha','beta']:
        rel = bp[band] / total
        for ch, ch_name in enumerate(CHANNEL_NAMES):
            features.append(rel[ch])
            names.append(f'rel_{band}_{ch_name}')

    # Frontal theta / Parietal alpha ratio  (PCLI proxy del proposal)
    front_idx = [CHANNEL_NAMES.index(c) for c in ['AF3','F3','F4','AF4','FC5','FC6']]
    parie_idx = [CHANNEL_NAMES.index(c) for c in ['P7','P8']]
    ft = bp['theta'][front_idx].mean()
    pa = bp['alpha'][parie_idx].mean()
    features.append(np.log1p(ft / (pa + 1e-10)))
    names.append('frontal_theta_parietal_alpha_ratio')

    for ch, ch_name in enumerate(CHANNEL_NAMES):
        pn = psd[:,ch] / (psd[:,ch].sum() + 1e-10)
        features.append(entropy(pn + 1e-10))
        names.append(f'spectral_entropy_{ch_name}')

    for ch, ch_name in enumerate(CHANNEL_NAMES):
        features.append(np.var(epoch[:,ch]))
        names.append(f'variance_{ch_name}')
    for ch, ch_name in enumerate(CHANNEL_NAMES):
        features.append(skew(epoch[:,ch]))
        names.append(f'skewness_{ch_name}')

    return np.array(features), names

def extract_windows(eeg, label):
    step = int(WINDOW_SAMPLES * (1 - OVERLAP))
    X, y = [], []
    for start in range(0, eeg.shape[0] - WINDOW_SAMPLES + 1, step):
        feats, _ = extract_features(eeg[start:start+WINDOW_SAMPLES])
        X.append(feats)
        y.append(label)
    return np.array(X), np.array(y)

print("✓ Funciones de feature extraction definidas")

## Celda 8 — Construir la matriz de features
Esto procesa todos los sujetos. Tarda ~2-4 minutos con los 48 sujetos reales.

In [ ]:
all_X, all_y, all_groups = [], [], []
feat_names = None

for subj in subjects:
    rest_eeg = preprocess_eeg(subj['rest'])
    task_eeg = preprocess_eeg(subj['task'])
    X_rest, y_rest = extract_windows(rest_eeg, 0)
    X_task, y_task = extract_windows(task_eeg, 1)
    if feat_names is None:
        _, feat_names = extract_features(rest_eeg[:WINDOW_SAMPLES])
    all_X.append(np.vstack([X_rest, X_task]))
    all_y.append(np.concatenate([y_rest, y_task]))
    all_groups.extend([subj['id']] * (len(y_rest) + len(y_task)))
    print(f"  {subj['id']}: {len(y_rest)} rest + {len(y_task)} task windows")

X = np.nan_to_num(np.vstack(all_X))
y = np.concatenate(all_y)
groups = np.array(all_groups)

print(f"\n✓ Dataset completo: {X.shape[0]} ventanas x {X.shape[1]} features")
print(f"  Balance: {(y==0).sum()} REST | {(y==1).sum()} TASK")

## Celda 9 — Entrenar y evaluar modelos
SVM, Random Forest y Logistic Regression con 5-fold cross-validation.

In [ ]:
models = {
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=1.0, probability=True, random_state=SEED))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, max_depth=10,
                                        random_state=SEED, n_jobs=-1))
    ]),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED))
    ])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = {}

for name, model in models.items():
    print(f"Entrenando {name}...")
    y_pred  = cross_val_predict(model, X, y, cv=cv, method='predict')
    y_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:,1]
    results[name] = {
        'accuracy': accuracy_score(y, y_pred),
        'f1':       f1_score(y, y_pred),
        'auc':      roc_auc_score(y, y_proba),
        'y_pred':   y_pred,
        'y_proba':  y_proba
    }
    print(f"  Accuracy: {results[name]['accuracy']:.3f} | "
          f"F1: {results[name]['f1']:.3f} | "
          f"ROC-AUC: {results[name]['auc']:.3f}")

print("\n✓ Entrenamiento completo")

## Celda 10 — Feature Importance (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier as RFC
rf = RFC(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(StandardScaler().fit_transform(X), y)

importance_df = pd.DataFrame({
    'feature':    feat_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print("Top 10 features más importantes:")
print(importance_df.head(10).to_string(index=False))

## Celda 11 — Visualizaciones
Genera la figura de resultados con 4 paneles.

In [ ]:
fig = plt.figure(figsize=(16, 12))
fig.suptitle('EEG Mental Workload Classification — STEW Dataset\nINEL 4998 | UPRM AIIG Lab',
             fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)
colors = {'SVM (RBF)':'#2196F3','Random Forest':'#4CAF50','Logistic Regression':'#FF9800'}

# Panel A: Model comparison
ax1 = fig.add_subplot(gs[0,0])
metrics = ['accuracy','f1','auc']
labels  = ['Accuracy','F1 Score','ROC-AUC']
x = np.arange(3)
for i, (name, res) in enumerate(results.items()):
    vals = [res[m] for m in metrics]
    bars = ax1.bar(x + i*0.25, vals, 0.25, label=name, color=colors[name], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{v:.2f}', ha='center', va='bottom', fontsize=8)
ax1.set_xticks(x+0.25); ax1.set_xticklabels(labels)
ax1.set_ylim(0,1.15); ax1.set_title('Comparación de Modelos')
ax1.legend(fontsize=8); ax1.grid(axis='y',alpha=0.3)
ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.4)

# Panel B: Confusion matrix (mejor modelo)
ax2 = fig.add_subplot(gs[0,1])
best = max(results, key=lambda k: results[k]['auc'])
cm = confusion_matrix(y, results[best]['y_pred']).astype(float)
cm /= cm.sum(axis=1)[:,None]
sns.heatmap(cm, annot=True, fmt='.2f', ax=ax2, cmap='Blues',
            xticklabels=['REST','TASK'], yticklabels=['REST','TASK'])
ax2.set_title(f'Confusion Matrix — {best}')
ax2.set_ylabel('True'); ax2.set_xlabel('Predicted')

# Panel C: Feature importance top 20
ax3 = fig.add_subplot(gs[1,0])
top = importance_df.head(20)
feat_colors = ['#E91E63' if 'theta' in f else
               '#9C27B0' if 'alpha' in f else
               '#3F51B5' if 'beta'  in f else
               '#FF5722' if 'ratio' in f else
               '#607D8B' for f in top['feature']]
ax3.barh(range(20), top['importance'], color=feat_colors, alpha=0.85)
ax3.set_yticks(range(20)); ax3.set_yticklabels(top['feature'], fontsize=7)
ax3.invert_yaxis(); ax3.set_title('Top 20 Features (Random Forest)')
ax3.set_xlabel('Importancia'); ax3.grid(axis='x', alpha=0.3)
from matplotlib.patches import Patch
ax3.legend(handles=[Patch(facecolor='#E91E63',label='Theta'),
                    Patch(facecolor='#9C27B0',label='Alpha'),
                    Patch(facecolor='#3F51B5',label='Beta'),
                    Patch(facecolor='#FF5722',label='Ratio')], fontsize=7)

# Panel D: Band power REST vs TASK
ax4 = fig.add_subplot(gs[1,1])
band_names = ['delta','theta','alpha','beta','gamma']
rest_ep = preprocess_eeg(subjects[0]['rest'])[:WINDOW_SAMPLES]
task_ep = preprocess_eeg(subjects[0]['task'])[:WINDOW_SAMPLES]
bp_r, _, _ = compute_band_power(rest_ep)
bp_t, _, _ = compute_band_power(task_ep)
x2 = np.arange(len(band_names))
ax4.bar(x2-0.2, [np.log1p(bp_r[b].mean()) for b in band_names], 0.4,
        label='REST', color='#42A5F5', alpha=0.85)
ax4.bar(x2+0.2, [np.log1p(bp_t[b].mean()) for b in band_names], 0.4,
        label='TASK', color='#EF5350', alpha=0.85)
ax4.set_xticks(x2); ax4.set_xticklabels([b.capitalize() for b in band_names])
ax4.set_title('Band Power: REST vs TASK (sub01)'); ax4.legend()
ax4.set_ylabel('Log Power (µV²/Hz)'); ax4.grid(axis='y',alpha=0.3)

plt.savefig('results_figure.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Figura guardada como results_figure.png")

## Celda 12 — Resumen final y guardar resultados

In [ ]:
print("="*60)
print("  RESULTADOS FINALES")
print("="*60)
print(f"{'Modelo':<22} {'Accuracy':>10} {'F1':>10} {'ROC-AUC':>10}")
print("-"*55)
best = max(results, key=lambda k: results[k]['auc'])
for name, res in results.items():
    marker = " <- mejor" if name == best else ""
    print(f"{name:<22} {res['accuracy']:>10.3f} {res['f1']:>10.3f} {res['auc']:>10.3f}{marker}")

print("\nTop 5 features:")
for i, row in importance_df.head(5).iterrows():
    print(f"  {i+1}. {row['feature']:<40} {row['importance']:.4f}")

# Guardar CSVs
pd.DataFrame([{'Model':n,'Accuracy':r['accuracy'],'F1':r['f1'],'ROC_AUC':r['auc']}
               for n,r in results.items()]).to_csv('model_results.csv', index=False)
importance_df.to_csv('feature_importance.csv', index=False)
print("\n✓ Guardado: model_results.csv, feature_importance.csv, results_figure.png")
print("  (Para descargarlos: panel izquierdo de Colab > icono de carpeta)")